# 03 — Cross-Standard GRA Message Translation

Translates defense messages across OMS/UCI, FACE, and Link 16
using the graph-native holonic architecture with UDDL semantic
layering and CCO-grounded conceptual alignment.

This example uses the redesigned `HolonicDataset` where:
- The dataset IS the holarchy (no separate registry object)
- Portals are RDF in boundary graphs, discovered via SPARQL
- Multi-source CONSTRUCT queries scope data with GRAPH clauses
- Membrane validation operates on the union of interior graphs

In [ ]:
from holonic import HolonicDataset, MembraneHealth

ds = HolonicDataset()

# ══════════════════════════════════════════════════════════════
# PHASE 1: UDDL Conceptual Data Model (semantic commons)
# ══════════════════════════════════════════════════════════════

ds.add_holon("urn:holon:uddl:cdm", "UDDL Conceptual Data Model")
ds.add_interior("urn:holon:uddl:cdm", """
    @prefix uddl:     <https://ghost-core.tail515200.ts.net/ontology/uddl#> .
    @prefix uddl-cdm: <urn:uddl:cdm:> .
    @prefix uddl-obs: <urn:uddl:observable:> .
    @prefix skos:     <http://www.w3.org/2004/02/skos/core#> .
    @prefix cco:      <https://www.commoncoreontologies.org/> .

    uddl-cdm:PlatformPosition a uddl:ConceptualEntity ;
        uddl:name "PlatformPosition" ;
        uddl:description "Geospatial position of a platform." ;
        skos:exactMatch cco:GeospatialPosition .

    uddl-cdm:MissionTaskAssignment a uddl:ConceptualEntity ;
        uddl:name "MissionTaskAssignment" ;
        uddl:description "Directive assigning a unit to a mission." ;
        skos:exactMatch cco:ActOfDirectiveCommunication .

    uddl-obs:Latitude a uddl:Observable ;
        uddl:name "Latitude" ;
        uddl:description "Angular distance from equator." .
    uddl-obs:Longitude a uddl:Observable ;
        uddl:name "Longitude" ;
        uddl:description "Angular distance from prime meridian." .
""")

# ══════════════════════════════════════════════════════════════
# PHASE 2: OMS Track Message holon
# ══════════════════════════════════════════════════════════════

ds.add_holon("urn:holon:oms-track", "OMS/UCI Track Message",
             member_of="urn:holon:uddl:cdm")

ds.add_interior("urn:holon:oms-track", """
    @prefix oms: <urn:standard:oms:> .
    <urn:msg:oms:track-001> a oms:TrackReport ;
        oms:trackNumber "TN-4401" ;
        oms:latitude 34.05 ;
        oms:longitude -118.25 ;
        oms:altitude 10000.0 ;
        oms:timestamp "2026-03-31T14:22:00Z"^^xsd:dateTime ;
        oms:trackQuality 12 .
    <urn:msg:oms:track-002> a oms:TrackReport ;
        oms:trackNumber "TN-4402" ;
        oms:latitude 34.12 ;
        oms:longitude -118.19 ;
        oms:altitude 8500.0 ;
        oms:timestamp "2026-03-31T14:22:05Z"^^xsd:dateTime ;
        oms:trackQuality 9 .
""")

ds.add_boundary("urn:holon:oms-track", """
    @prefix oms: <urn:standard:oms:> .
    <urn:shapes:oms:TrackShape> a sh:NodeShape ;
        sh:targetClass oms:TrackReport ;
        sh:property [
            sh:path oms:latitude ;
            sh:minCount 1 ;
            sh:datatype xsd:decimal ;
            sh:minInclusive -90.0 ;
            sh:maxInclusive 90.0 ;
            sh:severity sh:Violation
        ] ;
        sh:property [
            sh:path oms:longitude ;
            sh:minCount 1 ;
            sh:datatype xsd:decimal ;
            sh:severity sh:Violation
        ] ;
        sh:property [
            sh:path oms:timestamp ;
            sh:minCount 1 ;
            sh:severity sh:Violation
        ] .
""")

ds.add_projection("urn:holon:oms-track", """
    @prefix oms:      <urn:standard:oms:> .
    @prefix uddl:     <https://ghost-core.tail515200.ts.net/ontology/uddl#> .
    @prefix uddl-ldm: <urn:uddl:ldm:> .
    oms:TrackReport uddl:realizes uddl-ldm:WGS84PositionMeasurement .
""")

# ══════════════════════════════════════════════════════════════
# PHASE 3: FACE PDM Position holon (target — starts empty)
# ══════════════════════════════════════════════════════════════

ds.add_holon("urn:holon:face-pos", "FACE PDM AircraftPosition",
             member_of="urn:holon:uddl:cdm")

ds.add_boundary("urn:holon:face-pos", """
    @prefix face: <urn:standard:face:> .
    <urn:shapes:face:PositionShape> a sh:NodeShape ;
        sh:targetClass face:AircraftPosition ;
        sh:property [
            sh:path face:lat ;
            sh:minCount 1 ;
            sh:datatype xsd:double ;
            sh:minInclusive -90.0 ;
            sh:maxInclusive 90.0 ;
            sh:severity sh:Violation
        ] ;
        sh:property [
            sh:path face:lon ;
            sh:minCount 1 ;
            sh:datatype xsd:double ;
            sh:severity sh:Violation
        ] ;
        sh:property [
            sh:path face:timestamp ;
            sh:minCount 1 ;
            sh:severity sh:Violation
        ] .
""")

ds.add_projection("urn:holon:face-pos", """
    @prefix face:     <urn:standard:face:> .
    @prefix uddl:     <https://ghost-core.tail515200.ts.net/ontology/uddl#> .
    @prefix uddl-ldm: <urn:uddl:ldm:> .
    face:AircraftPosition uddl:realizes uddl-ldm:WGS84PositionMeasurement .
""")

# ══════════════════════════════════════════════════════════════
# PHASE 4: Alignment axioms holon
# ══════════════════════════════════════════════════════════════

ds.add_holon("urn:holon:alignment", "Position Alignment Axioms")
ds.add_interior("urn:holon:alignment", """
    @prefix oms:      <urn:standard:oms:> .
    @prefix face:     <urn:standard:face:> .
    @prefix uddl-obs: <urn:uddl:observable:> .

    uddl-obs:hasLatitudeValue  a rdf:Property .
    uddl-obs:hasLongitudeValue a rdf:Property .
    uddl-obs:hasAltitudeValue  a rdf:Property .
    uddl-obs:hasTimestampValue a rdf:Property .

    oms:latitude  rdfs:subPropertyOf uddl-obs:hasLatitudeValue .
    face:lat      rdfs:subPropertyOf uddl-obs:hasLatitudeValue .
    oms:longitude rdfs:subPropertyOf uddl-obs:hasLongitudeValue .
    face:lon      rdfs:subPropertyOf uddl-obs:hasLongitudeValue .
    oms:altitude  rdfs:subPropertyOf uddl-obs:hasAltitudeValue .
    face:alt_m    rdfs:subPropertyOf uddl-obs:hasAltitudeValue .
    oms:timestamp rdfs:subPropertyOf uddl-obs:hasTimestampValue .
    face:timestamp rdfs:subPropertyOf uddl-obs:hasTimestampValue .
""")

# ══════════════════════════════════════════════════════════════
# PHASE 5: Portal — OMS → FACE (multi-source CONSTRUCT)
# ══════════════════════════════════════════════════════════════

# The CONSTRUCT scopes source data with GRAPH clauses.
# The alignment holon is referenced for semantic justification.
ds.add_portal(
    "urn:portal:oms-to-face-pos",
    "urn:holon:oms-track",
    "urn:holon:face-pos",
    label="OMS Track → FACE Position",
    construct_query="""
        PREFIX oms:  <urn:standard:oms:>
        PREFIX face: <urn:standard:face:>
        CONSTRUCT {
            ?s a face:AircraftPosition ;
                face:lat       ?lat ;
                face:lon       ?lon ;
                face:alt_m     ?alt ;
                face:timestamp ?ts .
        }
        WHERE {
            GRAPH <urn:holon:oms-track/interior> {
                ?s a oms:TrackReport ;
                    oms:latitude  ?lat ;
                    oms:longitude ?lon ;
                    oms:timestamp ?ts .
                OPTIONAL { ?s oms:altitude ?alt }
            }
        }
    """,
)

print(ds.summary())

## Traverse: OMS → FACE with validation and provenance

In [ ]:
projected, membrane_result = ds.traverse(
    "urn:holon:oms-track",
    "urn:holon:face-pos",
    inject=True,
    validate=True,
    agent_iri="urn:agent:gra-pipeline-v2",
)

print(f"Projected {len(projected)} triples")
print()
print(membrane_result.summary())

## Query the translated data via SPARQL

In [ ]:
rows = ds.query("""
    PREFIX face: <urn:standard:face:>
    SELECT ?s ?lat ?lon ?alt ?ts
    WHERE {
        GRAPH <urn:holon:face-pos/interior> {
            ?s a face:AircraftPosition ;
                face:lat ?lat ;
                face:lon ?lon ;
                face:timestamp ?ts .
            OPTIONAL { ?s face:alt_m ?alt }
        }
    }
    ORDER BY ?ts
""")
print("FACE AircraftPosition records:")
for r in rows:
    print(f"  {r['s'].rsplit(':',1)[-1]:30s}  "
          f"lat={r['lat']}  lon={r['lon']}  alt={r.get('alt','—')}")

## Inspect provenance

In [ ]:
g = ds.backend.get_graph("urn:holon:face-pos/context")
print(f"Provenance trail ({len(g)} triples):")
for s, p, o in sorted(g):
    print(f"  {str(s).rsplit(':',1)[-1][:35]:37s} "
          f"{str(p).rsplit('/',1)[-1].rsplit('#',1)[-1]:25s} "
          f"{str(o)[:50]}")

## Verify semantic equivalence via projections

Both platform holons realize the same UDDL logical entity.
This is queryable — not hardcoded in Python.

In [ ]:
rows = ds.query("""
    PREFIX uddl: <https://ghost-core.tail515200.ts.net/ontology/uddl#>
    SELECT ?platform_class ?logical_entity
    WHERE {
        GRAPH ?g { ?platform_class uddl:realizes ?logical_entity }
    }
""")
print("UDDL realization links:")
for r in rows:
    cls = r["platform_class"].rsplit(":", 1)[-1]
    ldm = r["logical_entity"].rsplit(":", 1)[-1]
    print(f"  {cls:30s} realizes  {ldm}")

## Visualization

In [ ]:
from holonic.viz import HolonViz, HolarchyViz

In [ ]:
hz = HolonViz(ds, holon_iri="urn:holon:uddl:cdm")
hz.show()

In [ ]:
hz = HolarchyViz(ds, show_internals=True)
hz.show()